# Criba de motores TTS — Chatterbox Multilingual (V2, V3 y pack es-es)

Tres variantes del mismo motor, una detrás de otra: **V2** (la que ganó tu criba de junio), **V3** y **V3 con el pack de español de España** (`ResembleAI/Chatterbox-Multilingual-es-es`). Parámetros por defecto de la librería: `exaggeration=0.5`, `cfg_weight=0.5`, `temperature=0.8`.

**GPU:** T4. **Tiempo estimado:** unos 60 min, incluida la instalación.

**Cómo se usa** (es igual en todos los cuadernos de la criba):

1. *Entorno de ejecución → Cambiar tipo de entorno* → **T4**.
2. Ejecuta la **celda 1** (instalación). Cuando termine, *Entorno de ejecución → Reiniciar sesión*.
3. Ejecuta las demás celdas **en orden**, de la 2 a la última. La celda 2 comprueba que estén las
   referencias y las frases antes de cargar ningún modelo: si dice `RESULTADO: REVISAR`, para ahí.
4. La última celda **desconecta la GPU sola** para que no siga gastando unidades.

Si la sesión se corta a medias, vuelve a empezar por la celda 1: lo que ya esté en Drive no se
genera otra vez.

Salida: `fine_tunned_naval/outputs/criba/<variante>/` en Drive, un `.wav` por item (`n01`, `v01`,
`b01`, `k01`…) más `registro.jsonl`, `parametros.json` y `entorno.json`.

In [ ]:
# 1. Instalación (después: Entorno de ejecución → Reiniciar sesión)
# chatterbox-tts fija torch 2.6.0, transformers 5.2.0 y numpy<2: por eso hay que reiniciar.
# Se instala desde GitHub: la 0.1.7 de PyPI es anterior a la V3 y solo carga la V2.
# torchvision se fija a la versión que casa con torch 2.6.0: la que trae Colab pide otro torch.
# El aviso rojo "pip's dependency resolver..." que sale después es normal.
!pip install -q "chatterbox-tts @ git+https://github.com/resemble-ai/chatterbox.git@master" torchvision==0.21.0

In [ ]:
# 2. Preparar: código del repo, Drive, frases y referencias
import os, sys, subprocess
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path.insert(0, "/content/repo/criba")
import comun

print(sys.version.split()[0], "|", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                                                  capture_output=True, text=True).stdout.strip())
comun.montar_drive()
ITEMS, SILENCIO_MS = comun.cargar_items(None)
REFS = comun.comprobar(ITEMS)

In [ ]:
# 3. Funciones de carga y síntesis
import os, shutil, torch
from pathlib import Path
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
from huggingface_hub import snapshot_download

INFO = {}

def cargar(variante):
    if variante == "chatterbox_v2":
        return ChatterboxMultilingualTTS.from_pretrained(device="cuda", t3_model="v2")
    if variante == "chatterbox_v3":
        return ChatterboxMultilingualTTS.from_pretrained(device="cuda", t3_model="v3")
    if variante == "chatterbox_v3_es":
        # El pack es un fine-tune de T3. Se combina con las piezas comunes del repo base.
        base = Path(snapshot_download("ResembleAI/chatterbox", allow_patterns=[
            "ve.pt", "s3gen.pt", "grapheme_mtl_merged_expanded_v1.json", "conds.pt", "Cangjie5_TC.json"]))
        pack = Path(snapshot_download("ResembleAI/Chatterbox-Multilingual-es-es"))
        print("Ficheros del pack es-es:", sorted(p.name for p in pack.rglob("*") if p.is_file()))
        if (pack / "README.md").exists():
            print("--- README del pack (primeras líneas) ---")
            print("\n".join((pack / "README.md").read_text(encoding="utf-8").splitlines()[:40]))
            print("---")
        junto = Path("/content/chatterbox_es_es")
        junto.mkdir(exist_ok=True)
        for origen in (base, pack):
            for p in origen.rglob("*"):
                if p.is_file() and p.suffix in {".pt", ".json", ".safetensors"}:
                    shutil.copy(p, junto / p.name)
        t3s = sorted(p.name for p in pack.rglob("t3*.safetensors"))
        assert t3s, "El pack no trae ningún t3*.safetensors: revisa la lista de ficheros de arriba"
        t3 = t3s[0]
        print("T3 del pack:", t3)
        # El pack trae su propio decodificador (s3gen_v3). Se intenta con él; si no casa con
        # el código instalado, se vuelve al s3gen base y queda anotado en parametros.json.
        if (pack / "s3gen_v3.pt").exists():
            shutil.copy(pack / "s3gen_v3.pt", junto / "s3gen.pt")
            try:
                INFO[variante] = {"t3": t3, "s3gen": "s3gen_v3.pt del pack"}
                return ChatterboxMultilingualTTS.from_local(junto, "cuda", t3_model=t3)
            except Exception as e:
                print(f"s3gen_v3 del pack no carga ({type(e).__name__}: {str(e)[:200]}); pruebo con el s3gen base")
                shutil.copy(base / "s3gen.pt", junto / "s3gen.pt")
        INFO[variante] = {"t3": t3, "s3gen": "s3gen.pt base"}
        return ChatterboxMultilingualTTS.from_local(junto, "cuda", t3_model=t3)
    raise ValueError(variante)

PARAMS = dict(exaggeration=0.5, cfg_weight=0.5, temperature=0.8)

def hacer_sintetizar(modelo):
    ref_cargada = {"wav": None}
    def sintetizar(texto, ref):
        if ref_cargada["wav"] != ref["wav"]:
            modelo.prepare_conditionals(ref["wav"], exaggeration=PARAMS["exaggeration"])
            ref_cargada["wav"] = ref["wav"]
        wav = modelo.generate(texto, language_id="es", **PARAMS)
        return wav, modelo.sr
    return sintetizar
print("OK")

In [ ]:
# 4. Generar las tres variantes (se salta lo que ya esté en Drive)
for variante in ["chatterbox_v2", "chatterbox_v3", "chatterbox_v3_es"]:
    print(f"\n=== {variante}")
    try:
        modelo = cargar(variante)
    except Exception as e:
        print(f"No se pudo cargar {variante}: {type(e).__name__}: {e}. Paso a la siguiente.")
        continue
    comun.guardar_entorno(variante, ["chatterbox-tts", "torch", "transformers", "numpy"])
    comun.generar(variante, hacer_sintetizar(modelo), ITEMS, REFS, SILENCIO_MS,
                  params={**PARAMS, "language_id": "es", **INFO.get(variante, {})})
    del modelo
    comun.liberar()

In [ ]:
# Última celda: resumen y desconexión de la GPU
import json
for variante in ['chatterbox_v2', 'chatterbox_v3', 'chatterbox_v3_es']:
    reg = comun.SALIDA / variante / "registro.jsonl"
    if not reg.exists():
        print(f"{variante}: sin registro")
        continue
    filas = [json.loads(l) for l in reg.read_text(encoding="utf-8").splitlines()]
    ok = sum(f["ok"] for f in filas)
    print(f"{variante}: {ok} items correctos, {len(filas) - ok} con error")
comun.desconectar()